# Lesson 11 · Capacitors

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/StativeSh/quantum-playground/blob/main/lessons/11_capacitors.ipynb)

**For:** A-level Physics (for example AQA 3.7.4, OCR A 6.1, Edexcel Topic 7) and IB Physics (HL capacitance and RC circuits). Check your own specification for the exact wording.

**Before this lesson:** Lesson 5, *Electric fields and potential* (potential difference, W = qV and the field between parallel plates); Lesson 10, *Radioactive decay* (exponential decay, half-life and finding a constant from a log graph). Also assumed: current, resistance and Kirchhoff's laws.

**Leads to:** the end of this route for now.

**You will be able to:**

1. use $Q = CV$ and $C = \varepsilon_0\varepsilon_rA/d$
2. calculate the energy stored, $E = \tfrac12QV = \tfrac12CV^2$, and explain it as the area under a $Q$–$V$ graph
3. use $V = V_0e^{-t/RC}$ for discharging and $V = V_0(1 - e^{-t/RC})$ for charging
4. explain the time constant $RC$, and find it from a graph of $\ln V$ against $t$
5. explain why half the energy from the supply is lost as heat when a capacitor charges through a resistor

**How to use this lesson:** run each cell with Shift+Enter. Where you see *Try it*, change a number and run the cell again. Before some experiments a **Predict** box asks you to commit to an answer; write it down, run the cell, then answer the **Explain** box below it. Answers to both are at the very end, after the worked solutions. Exercises are at the end, with worked solutions below them.

In [ ]:
# Setup: installs the qplay package if it is missing (for example on Google Colab).
import importlib.util, subprocess, sys
if importlib.util.find_spec("qplay") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "quantum-playground"], check=True)

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from qplay import circuits

EPSILON_0 = 8.85e-12      # F/m, permittivity of free space

## 1. Capacitance

A capacitor is two conductors separated by an insulator. Connect it to a supply and charge $+Q$ collects on one plate and $-Q$ on the other. The charge is proportional to the potential difference across it:

$$Q = CV,$$

where $C$ is the **capacitance**, in farads (F). One farad is huge: everyday capacitors are measured in µF, nF and pF.

For two parallel plates of area $A$ a distance $d$ apart, filled with a material of relative permittivity $\varepsilon_r$ (the dielectric),

$$C = \frac{\varepsilon_0\varepsilon_rA}{d}.$$

**Predict (P1).** A parallel-plate capacitor has $C = 1.0$ nF. If you halve the gap between the plates *and* fill the gap with a material of relative permittivity 3, what is the new capacitance? Write your prediction down before you run the next cell.

In [ ]:
A, d = 0.010, 0.0885e-3                # plate area in m^2 and gap in m: 1.0 nF in air   <- Try it
for label, gap, eps_r in [("air, gap d", d, 1.0), ("air, gap d/2", d / 2, 1.0), ("eps_r = 3, gap d/2", d / 2, 3.0)]:
    C = EPSILON_0 * eps_r * A / gap
    print(f"{label:22s} C = {C * 1e9:6.3f} nF")

V = np.linspace(0, 12, 50)
plt.figure(figsize=(6, 3.6))
for C, label in [(470e-6, "470 µF"), (220e-6, "220 µF")]:
    plt.plot(V, C * V * 1e3, label=label)
plt.xlabel("V / V"); plt.ylabel("Q / mC"); plt.title("Q = CV: the gradient is the capacitance"); plt.legend();

**Explain (P1).** Explain each change in terms of the formula, and say physically why a smaller gap stores more charge at the same voltage.

## 2. Energy stored

Moving charge onto a capacitor takes work, because the charge already there repels the next bit. When the voltage is $V$, adding a little charge $\Delta Q$ takes work $V\Delta Q$. So the total energy stored is the **area under the $Q$–$V$ graph**, a triangle:

$$E = \tfrac12QV = \tfrac12CV^2 = \frac{Q^2}{2C}.$$

In [ ]:
C, V0 = 470e-6, 9.0
V = np.linspace(0, V0, 1001)
Q = C * V
area = np.sum(0.5 * (V[1:] + V[:-1]) * np.diff(Q))       # add up V dQ (trapezium rule)
plt.figure(figsize=(6, 3.6))
plt.plot(V, Q * 1e3); plt.fill_between(V, Q * 1e3, alpha=0.25)
plt.xlabel("V / V"); plt.ylabel("Q / mC"); plt.title("Energy stored = area under the Q-V graph")
print(f"area under the graph = {area * 1e3:.3f} mJ;   CV^2/2 = {0.5 * C * V0**2 * 1e3:.3f} mJ")

**Exam tip:** the energy is $\tfrac12QV$, **not** $QV$. The charge did not all move across the full voltage $V$; the voltage built up from zero as the charge arrived.

## 3. Discharging through a resistor

Connect a charged capacitor across a resistor $R$. The current is $I = V/R = Q/RC$, and it reduces the charge: $\dfrac{dQ}{dt} = -\dfrac{Q}{RC}$. This is exactly the equation of radioactive decay, with $\lambda$ replaced by $1/RC$, so

$$Q = Q_0e^{-t/RC},\qquad V = V_0e^{-t/RC},\qquad I = I_0e^{-t/RC}.$$

$\tau = RC$ is the **time constant**, in seconds when $R$ is in ohms and $C$ in farads.

**Predict (P2).** After one time constant, $t = RC$, what percentage of the starting voltage is left: 50%, 37% or 0%? Write your prediction down before you run the next cell.

In [ ]:
R, C, V0 = 10e3, 470e-6, 9.0                  # ohms, farads, volts   <- Try it
tau = R * C
t, q, I = circuits.rc_circuit(R, C, V=0.0, q0=C * V0, t_max=5 * tau)    # integrates R dq/dt + q/C = 0
V = q / C

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
ax1.plot(t, V, label="simulated")
ax1.plot(t[::40], circuits.rc_discharge(t[::40], V0, R, C), "k.", label=r"$V_0e^{-t/RC}$")
ax1.axvline(tau, color="C3", lw=0.8); ax1.axvline(tau * np.log(2), color="C2", lw=0.8)
ax1.text(tau, V0 * 0.9, "  t = RC", color="C3"); ax1.text(tau * np.log(2), V0 * 0.75, "  t = RC ln2", color="C2")
ax1.set(xlabel="t / s", ylabel="V / V", title=f"Discharge: RC = {tau:.1f} s"); ax1.legend()
ax2.plot(t, np.log(V))
ax2.set(xlabel="t / s", ylabel="ln V", title=f"ln V is a straight line, gradient -1/RC = {-1 / tau:.4f} per s")
k = np.searchsorted(t, tau)
print(f"after t = RC = {tau:.2f} s: V = {V[k]:.3f} V = {100 * V[k] / V0:.1f}% of the start")
print(f"time to halve: RC ln2 = {tau * np.log(2):.2f} s")

**Explain (P2).** Why doesn't the voltage fall to half after one time constant, and why does it never reach zero?

Notice the link with radioactive decay: there, a **half-life** $t_{1/2} = \ln 2/\lambda$; here, the **time to halve** is $RC\ln 2 \approx 0.69RC$. The same log-graph method finds $RC$ from measured data.

## 4. Charging through a resistor

Now connect an uncharged capacitor and a resistor in series with a supply of e.m.f. $V_0$. At first the capacitor has no charge, so all of $V_0$ is across the resistor and the current is large, $I_0 = V_0/R$. As charge builds up, the capacitor's voltage rises, the resistor's falls, and the current dies away:

$$V_C = V_0\left(1 - e^{-t/RC}\right),\qquad V_R = V_0e^{-t/RC},\qquad I = \frac{V_0}{R}e^{-t/RC}.$$

In [ ]:
R, C, V0 = 10e3, 470e-6, 9.0
t, q, I = circuits.rc_circuit(R, C, V=V0, q0=0.0, t_max=5 * R * C)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
ax1.plot(t, q / C, label=r"$V_C$ (capacitor)")
ax1.plot(t, I * R, label=r"$V_R$ (resistor)")
ax1.plot(t, q / C + I * R, "k--", lw=1, label="sum = supply e.m.f.")
ax1.set(xlabel="t / s", ylabel="voltage / V"); ax1.legend()
ax2.plot(t, I * 1e3, color="C2")
ax2.set(xlabel="t / s", ylabel="I / mA", title="The current dies away as the capacitor fills");

At every moment $V_C + V_R = V_0$ (Kirchhoff's second law), which is why one curve rises exactly as the other falls.

## 5. Where does the energy go?

While charging, the supply moves charge $Q = CV_0$ through e.m.f. $V_0$, so it does work $QV_0 = CV_0^2$. The capacitor ends up storing $\tfrac12CV_0^2$.

**Predict (P3).** What fraction of the energy from the supply ends up stored in the capacitor? Does using a bigger resistor waste more energy, less, or the same? Write your prediction down before you run the next cell.

In [ ]:
C, V0 = 470e-6, 9.0
print(f"energy from the supply CV0^2 = {C * V0**2 * 1e3:.2f} mJ;  stored CV0^2/2 = {0.5 * C * V0**2 * 1e3:.2f} mJ")
for R in [100.0, 1e3, 10e3, 100e3]:                           # <- Try it: any resistance
    t, q, I = circuits.rc_circuit(R, C, V=V0, q0=0.0, t_max=12 * R * C, n=20001)
    heat = np.sum(0.5 * (I[1:] ** 2 + I[:-1] ** 2) * R * np.diff(t))   # add up I^2 R dt
    print(f"R = {R:>8.0f} ohm: charging takes ~{5 * R * C:7.2f} s, heat in the resistor = {heat * 1e3:.2f} mJ")

**Explain (P3).** A bigger resistor carries a smaller current. Why does it still turn the same amount of energy into heat?

---

## Exercises

Take $\varepsilon_0 = 8.85\times10^{-12}\ \mathrm{F\,m^{-1}}$. Give answers to 3 significant figures.

**Q1.** A 470 µF capacitor is charged to 9.0 V. Find (a) the charge on it, (b) the energy it stores.

**Q2.** Two parallel plates, each of area $0.010\ \mathrm{m^2}$, are 0.50 mm apart in air. Find the capacitance.

**Q3.** The capacitor in Q1 discharges through a 10 kΩ resistor. Find (a) the time constant, (b) the voltage after 10 s.

**Q4.** How long does the voltage in Q3 take to fall to half its starting value?

**Q5.** The same capacitor, uncharged, is connected to a 9.0 V supply through the 10 kΩ resistor. Find (a) the current at the moment it is connected, (b) the time for the capacitor's voltage to reach 6.0 V.

**Q6 (simulation).** How much energy is turned into heat in the resistor while the capacitor in Q5 charges fully? Check your answer with the code in section 5.

In [ ]:
def check(name, yours, correct, tol=0.01):
    if yours is None:
        print(f"{name}: not answered yet")
    elif abs(yours - correct) <= tol * abs(correct):
        print(f"{name}: correct")
    else:
        print(f"{name}: not yet, check your working (you gave {yours})")

# The correct values are computed here from the physics, not typed in.
_C, _V, _R = 470e-6, 9.0, 10e3
_answers = {
    "Q1a": _C * _V, "Q1b": 0.5 * _C * _V**2,
    "Q2": EPSILON_0 * 0.010 / 0.50e-3,
    "Q3a": _R * _C, "Q3b": float(circuits.rc_discharge(10.0, _V, _R, _C)),
    "Q4": _R * _C * np.log(2),
    "Q5a": _V / _R, "Q5b": -_R * _C * np.log(1 - 6.0 / _V),
    "Q6": _C * _V**2 - 0.5 * _C * _V**2,              # supply's work minus energy stored
}

# Replace None with your answers, in SI units.
mine = {"Q1a": None, "Q1b": None, "Q2": None, "Q3a": None, "Q3b": None, "Q4": None, "Q5a": None, "Q5b": None, "Q6": None}
for q in mine:
    check(q, mine[q], _answers[q])

---

## Worked solutions

Try the questions before reading on.

**Q1.** (a) $Q = CV = 470\times10^{-6} \times 9.0 = 4.23\times10^{-3}$ C &nbsp; (b) $E = \tfrac12CV^2 = \tfrac12 \times 470\times10^{-6} \times 81 = 1.90\times10^{-2}$ J

**Q2.** $C = \dfrac{\varepsilon_0A}{d} = \dfrac{8.85\times10^{-12} \times 0.010}{0.50\times10^{-3}} = 1.77\times10^{-10}$ F, i.e. 177 pF.

**Q3.** (a) $\tau = RC = 10\times10^3 \times 470\times10^{-6} = 4.70$ s &nbsp; (b) $V = 9.0\,e^{-10/4.70} = 1.07$ V

**Q4.** $\tfrac12 = e^{-t/RC} \Rightarrow t = RC\ln 2 = 4.70 \times 0.693 = 3.26$ s

**Q5.** (a) At first the capacitor has no voltage across it, so $I_0 = V_0/R = 9.0/10\,000 = 9.00\times10^{-4}$ A &nbsp; (b) $6.0 = 9.0\left(1 - e^{-t/RC}\right) \Rightarrow e^{-t/RC} = \tfrac13 \Rightarrow t = RC\ln 3 = 5.16$ s

**Q6.** The supply does work $QV_0 = CV_0^2 = 3.81\times10^{-2}$ J and the capacitor stores half of it, so the other half, $1.90\times10^{-2}$ J, becomes heat in the resistor, whatever its resistance.

In [ ]:
for q, value in _answers.items():
    print(f"{q}: {value:#.3g}")

**Next:** in section 3, change `R` to 1 kΩ and to 100 kΩ. How does the discharge curve change, and what stays the same on the log graph?

### Predict and explain: answers

**P1.** 6.0 nF. Halving $d$ doubles $C$ (to 2.0 nF), and the dielectric multiplies it by $\varepsilon_r = 3$. With the plates closer, the same voltage gives a stronger field between them ($E = V/d$), so more charge is needed on each plate to make it. A dielectric's molecules line up in the field and partly cancel it, so again more charge is needed for the same voltage.

**P2.** 37% ($e^{-1} = 0.368$). The voltage does not fall at a steady rate: the current is proportional to the voltage, so the capacitor discharges quickly at first and ever more slowly as the voltage drops. It halves after $RC\ln 2 \approx 0.69RC$, and then halves again in each further $0.69RC$, so it approaches zero but never reaches it.

**P3.** Exactly half is stored, and the other half becomes heat in the resistor, the same for every resistance. A bigger resistor carries a smaller current, $I_0 = V_0/R$, but for longer, since charging takes about $5RC$. The heat $\int I^2R\,dt$ scales as $R \times (V_0/R)^2 \times RC = CV_0^2$, with no $R$ left in it: the integral works out to exactly $\tfrac12CV_0^2$.